# UC-JL-2 — Manage Platform Notebooks (Sysadmin)

**Persona:** Platform Sysadmin

**Goal:** Publish a new "Dekadal Rainfall Composite Viewer" notebook platform-wide,
verify discoverability by tag, confirm that a regular admin token cannot overwrite it,
then delete it and confirm tenant copies are not affected.

**Prerequisites:**
- Platform running with the Notebooks extension registered
- Sysadmin JWT in `DYNASTORE_SYSADMIN_TOKEN`

**Env vars required:** `DYNASTORE_BASE_URL`, `DYNASTORE_SYSADMIN_TOKEN`  
**Optional:** `DYNASTORE_ADMIN_TOKEN` (used to demonstrate 403 in the edge-case cell)

In [ ]:
import os
import json

import httpx
from dotenv import load_dotenv

load_dotenv()

BASE_URL = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")
SYSADMIN_TOKEN = os.environ.get("DYNASTORE_SYSADMIN_TOKEN", "")

sysadmin_headers = {
    "Authorization": f"Bearer {SYSADMIN_TOKEN}",
    "Content-Type": "application/json",
}
client = httpx.Client(base_url=BASE_URL, headers=sysadmin_headers, timeout=120.0)

# Minimal platform notebook definition
platform_notebook = {
    "nbformat": 4,
    "nbformat_minor": 5,
    "metadata": {
        "kernelspec": {
            "display_name": "Python 3",
            "language": "python",
            "name": "python3",
        },
        "tags": ["rainfall", "dekadal"],
    },
    "cells": [
        {
            "cell_type": "markdown",
            "id": "a1b2c3d4",
            "metadata": {},
            "source": ["# Dekadal Rainfall Composite Viewer\n"],
        }
    ],
}
notebook_id = "dekadal-rainfall-viewer"

print(f"Connected to {BASE_URL}")
print(f"Publishing notebook_id={notebook_id}")

In [ ]:
# Guard: check whether notebook_id already exists before publishing
r = client.get("/notebooks/platform")
print(r.status_code)
assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text}"

body = r.json()
existing = body if isinstance(body, list) else body.get("notebooks", body.get("items", []))
existing_ids = [nb["notebook_id"] for nb in existing]
print(f"Existing platform notebook ids: {existing_ids}")

assert notebook_id not in existing_ids, (
    f"notebook_id '{notebook_id}' already exists on the platform. "
    f"Delete it first or choose a different id."
)
print(f"Confirmed: '{notebook_id}' not yet published.")

In [ ]:
# Publish the notebook to the platform namespace
r = client.put(
    f"/notebooks/platform/{notebook_id}",
    content=json.dumps(platform_notebook),
)
print(r.status_code, r.text[:400])
assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text}"

published = r.json()
print(f"Platform notebook published: {published}")

In [ ]:
# Verify direct retrieval by id
r = client.get(f"/notebooks/platform/{notebook_id}")
print(r.status_code)
assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text}"

fetched = r.json()
fetched_id = fetched.get("notebook_id", "")
assert fetched_id == notebook_id, f"id mismatch: expected '{notebook_id}', got '{fetched_id}'"

# Confirm the title is present in the first cell
cells = fetched.get("content", {}).get("cells", [])
first_src = "".join(cells[0].get("source", [])) if cells else ""
assert "Dekadal Rainfall" in first_src, (
    f"Expected 'Dekadal Rainfall' in first cell source, got: {first_src!r}"
)
print(f"Verified: notebook_id={fetched_id}, first cell title correct.")

In [ ]:
# Verify tag-based discoverability
r = client.get("/notebooks/platform", params={"tags": "rainfall", "limit": 5})
print(r.status_code)
assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text}"

body = r.json()
results = body if isinstance(body, list) else body.get("notebooks", body.get("items", []))
result_ids = [nb["notebook_id"] for nb in results]
print(f"Tag 'rainfall' results: {result_ids}")

assert notebook_id in result_ids, (
    f"notebook_id '{notebook_id}' not found in tag-filtered results: {result_ids}"
)
print(f"Confirmed: '{notebook_id}' is discoverable via tag=rainfall.")

## Edge Cases

In [ ]:
# Edge case 1: non-sysadmin tokens must be rejected when writing to the platform namespace.
#
# DYNASTORE_TOKEN is populated by the showcase conftest with the testuser JWT, which
# carries no sysadmin role; DYNASTORE_ADMIN_TOKEN is broadcast from the sysadmin token and
# would pass — do NOT use it for this check.
#
# Enforcement requires the IAM middleware + PermissionProtocol to be active. In a
# permissive dev stack (no authenticator wired) the route is reachable by any token and
# this cell prints a notice instead of asserting.
user_token = os.environ.get("DYNASTORE_TOKEN", "")
if not user_token:
    print("DYNASTORE_TOKEN unset; skipping non-sysadmin rejection check.")
else:
    user_headers = {
        "Authorization": f"Bearer {user_token}",
        "Content-Type": "application/json",
    }
    user_client = httpx.Client(base_url=BASE_URL, headers=user_headers, timeout=120.0)

    r = user_client.put(
        f"/notebooks/platform/{notebook_id}",
        content=json.dumps(platform_notebook),
    )
    print(r.status_code, r.text[:300])
    if r.status_code in (401, 403):
        print(f"{r.status_code} confirmed: only sysadmins may write to the platform namespace.")
    else:
        print(
            f"NOTE: write succeeded with status {r.status_code}. "
            "IAM middleware is in pass-through mode in this deployment; "
            "wire an AuthenticatorProtocol + PermissionProtocol to enforce 403."
        )
    user_client.close()

In [ ]:
# Delete the platform notebook
r = client.delete(f"/notebooks/platform/{notebook_id}")
print(r.status_code, r.text[:200])
assert r.status_code == 204, f"Expected 204, got {r.status_code}: {r.text}"
print(f"Platform notebook '{notebook_id}' deleted.")

# Confirm it is gone
r = client.get(f"/notebooks/platform/{notebook_id}")
print(r.status_code)
assert r.status_code == 404, (
    f"Expected 404 after deletion, got {r.status_code}: {r.text}"
)
print("404 confirmed: platform notebook is no longer accessible.")

In [ ]:
# Edge case 2: platform DELETE does not cascade to tenant copies
#
# When a sysadmin deletes a platform notebook, any copies that tenants previously
# created under /notebooks/{CATALOG_ID}/{notebook_code} are INDEPENDENT records in
# the tenant schema. They are not soft-deleted, shadowed, or orphaned.
#
# Verification pattern (requires a prior copy to exist):
#
#   copy_r = client.get(f"/notebooks/{catalog_id}/{copied_notebook_code}")
#   assert copy_r.status_code == 200, "Tenant copy unexpectedly missing after platform delete"
#
# This is the intended isolation behaviour: the platform namespace is read-only for
# tenants; once copied, the tenant owns the notebook and manages its own lifecycle.
#
# If the platform notebook served as a live template (not a copy), the reference
# becomes a dangling id after deletion — the platform should return 410 Gone on
# /notebooks/platform/{deleted_id} to allow clients to detect this condition.
print(
    "Platform notebook deletion does NOT affect tenant copies.\n"
    "Tenant copies are independent records; tenants own their lifecycle after POST /copy."
)
client.close()